# FAISS Vector Retrieval Notebook

This notebook is prepared for running retrieval after `index.faiss` and `payloads.jsonl` finish downloading.

**Architecture (001-faiss-retrieval-notebook):**
- Store: [`SQLitePayloadFaissVectorStore`](../src/retrieval/sqlite_faiss_store.py) — FAISS + rebuild-if-stale `payload_cache.sqlite`
- Generation: [`generation.reasoning_client`](../src/generation/reasoning_client.py) — OpenAI-compatible client with three-way reasoning parse

Expected index directory layout:

```text
data/faiss_index/
  index.faiss
  payloads.jsonl
  id_map.json        # optional, but recommended if available
  payload_cache.sqlite  # built automatically on first load
```

Run the cells from top to bottom. If downloads are not finished yet, the preflight cell will tell you what is still missing.

**Note:** Answer generation here is a thin demonstration/validation layer (not a full judge/scoring pipeline). For judged end-to-end evaluation, use `scripts/evaluate_e2e.py`.


## 1. Environment setup

In [ ]:
# Optional: install runtime dependencies if your environment does not have them yet.
# Uncomment and run once if needed.
%pip install -q faiss-cpu sentence-transformers pandas openai


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import logging

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # Useful if the notebook is launched from notebooks/.
    PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('HF Hub token detected in environment.')
else:
    logging.getLogger('huggingface_hub.utils._http').setLevel(logging.ERROR)
    print('HF_TOKEN not set; suppressing the Hugging Face unauthenticated-request warning.')

print('Project root:', PROJECT_ROOT)
print('src on path:', SRC_DIR.exists())


Project root: d:\Uni_Project\Text_Mining\Project
src on path: True


## 2. Configure artifact paths and retrieval settings

Change `INDEX_DIR` if you download the FAISS files somewhere else.

In [ ]:
# Directory containing index.faiss + payloads.jsonl (+ optional id_map.json)
INDEX_DIR = PROJECT_ROOT / 'data' / 'faiss_index'

# Must match the embedding model used to build index.faiss.
# Alias EMBEDDING_MODEL_NAME kept for plan/quickstart naming parity.
EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'
EMBEDDING_MODEL_NAME = EMBEDDING_MODEL

TOP_K = 30                 # candidates pulled from FAISS before reranking/dedup
TOP_N = 10                 # final chunks returned
SCORE_THRESHOLD = 0.30
EXPAND_UNITS = False       # set True to demo same-provision expansion (FR-009)
DEFAULT_FILTER_PROFILE = 'broad'  # current_law | broad | historical
FILTER_PROFILE = DEFAULT_FILTER_PROFILE
# graph_guided is NOT exercised here — requires the knowledge graph module (FR-010).
BENCHMARK_SAMPLE_SIZE = 10

INDEX_DIR


## 3. Preflight: wait until downloads are complete

In [ ]:
phase_t0 = time.perf_counter()
required_files = [INDEX_DIR / 'index.faiss', INDEX_DIR / 'payloads.jsonl']
optional_files = [INDEX_DIR / 'id_map.json']

missing = [p for p in required_files if not p.exists()]
if missing:
    print('Downloads are not ready yet. Missing:')
    for p in missing:
        print(' -', p)
else:
    print('Required files found.')
    for p in required_files + optional_files:
        if p.exists():
            print(f'{p.name}: {p.stat().st_size / 1024 / 1024:.2f} MB')
        else:
            print(f'{p.name}: not found (optional)')

print(f'Preflight completed in {time.perf_counter() - phase_t0:.2f}s')


Required files found.
index.faiss: 5911.63 MB
payloads.jsonl: 4836.18 MB
id_map.json: 60.91 MB


## 4. Load the FAISS store and build retriever

In [ ]:
if missing:
    raise FileNotFoundError('Download index.faiss and payloads.jsonl before running this cell.')

from retrieval.config import VectorIndexConfig
from retrieval.embeddings import SentenceTransformerEmbedder
from retrieval.retriever import VectorRetriever
from retrieval.sqlite_faiss_store import SQLitePayloadFaissVectorStore

load_t0 = time.perf_counter()
config = VectorIndexConfig(
    embedding_model=EMBEDDING_MODEL,
    top_k=TOP_K,
    top_n=TOP_N,
    score_threshold=SCORE_THRESHOLD,
    expand_units=EXPAND_UNITS,
)

store = SQLitePayloadFaissVectorStore.load(INDEX_DIR)
embedder = SentenceTransformerEmbedder(
    EMBEDDING_MODEL,
    query_prefix=config.query_prefix,
    passage_prefix=config.passage_prefix,
)
retriever = VectorRetriever(config=config, embedder=embedder, store=store)

print(f'Vector retriever ready in {time.perf_counter() - load_t0:.2f}s')
print(f'Loaded FAISS vectors: {store.total_vectors:,}')
print(f'Embedding dimension: {embedder.dimension}')
print('Store class:', type(store).__module__ + '.' + type(store).__name__)


## 4.1 Optional: export payload cache to CSV (for inspection)

In [ ]:
import csv


def export_payloads_to_csv(csv_path: Path | None = None, limit: int | None = 5000) -> Path:
    """Export the SQLite payloads table to CSV for manual inspection.

    Defaults to the first `limit` rows to keep the CSV small and fast to open.
    Pass limit=None to export the full table (this can be as large as payloads.jsonl).
    """
    csv_path = csv_path or (INDEX_DIR / 'payloads_export.csv')
    t0 = time.perf_counter()

    query = 'SELECT line_no, payload FROM payloads ORDER BY line_no'
    params: tuple = ()
    if limit is not None:
        query += ' LIMIT ?'
        params = (limit,)

    rows = []
    for line_no, payload_text in store._conn.execute(query, params):
        payload = json.loads(payload_text)
        rows.append({'line_no': line_no, **payload})

    # Payload schemas can vary slightly between chunks, so build the CSV header
    # from the union of keys seen across the exported rows.
    fieldnames: list[str] = []
    seen: set[str] = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                fieldnames.append(key)

    with csv_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

    print(f'Exported {len(rows):,} rows to {csv_path} in {time.perf_counter() - t0:.2f}s')
    return csv_path


export_payloads_to_csv()


## 4.2 Optional: download / export payload_cache.sqlite

In [ ]:
def export_payload_cache_sqlite(dest_dir: Path | None = None) -> Path:
    """Copy payload_cache.sqlite out of INDEX_DIR and offer it for download.

    - In Google Colab: triggers a browser download via `google.colab.files.download`.
    - If Google Drive is mounted at /content/drive: also copies the file there.
    - Otherwise: copies the file to `dest_dir` (defaults to the project root) so it is
      easy to locate for a manual download/copy.
    """
    import shutil

    src_path = store.cache_path
    if not src_path.exists():
        raise FileNotFoundError(f'payload_cache.sqlite not found at {src_path}')

    t0 = time.perf_counter()
    print(f'Source: {src_path} ({src_path.stat().st_size / 1024 / 1024:.2f} MB)')

    try:
        from google.colab import files as colab_files  # type: ignore
        in_colab = True
    except ImportError:
        colab_files = None
        in_colab = False

    drive_root = Path('/content/drive')
    if in_colab and drive_root.exists():
        drive_dest = drive_root / 'MyDrive' / 'faiss_payload_cache' / src_path.name
        drive_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, drive_dest)
        print(f'Copied to Google Drive: {drive_dest} in {time.perf_counter() - t0:.2f}s')
        return drive_dest

    if in_colab and colab_files is not None:
        print('Triggering browser download via Colab...')
        colab_files.download(str(src_path))
        print(f'Download triggered in {time.perf_counter() - t0:.2f}s')
        return src_path

    dest_dir = dest_dir or PROJECT_ROOT
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / src_path.name
    if dest_path.resolve() != src_path.resolve():
        shutil.copy2(src_path, dest_path)
    print(f'Copied to {dest_path} in {time.perf_counter() - t0:.2f}s')
    return dest_path


export_payload_cache_sqlite()


## 5. Retrieval helper

In [ ]:
def search(query: str, top_n: int = TOP_N, filter_profile: str = FILTER_PROFILE, score_threshold: float | None = SCORE_THRESHOLD, expand_units: bool | None = None):
    """Run VectorRetriever and return (display_rows, RetrievalResult).

    filter_profile: 'current_law' | 'broad' | 'historical'
    (graph_guided is out of scope for this notebook — requires the knowledge graph module.)
    """
    if filter_profile == 'graph_guided':
        print('graph_guided is not exercised here — requires knowledge graph module (FR-010). Falling back to broad.')
        filter_profile = 'broad'

    search_t0 = time.perf_counter()
    result = retriever.retrieve(
        query,
        top_n=top_n,
        filter_profile=filter_profile,
        score_threshold=score_threshold,
        expand_units=EXPAND_UNITS if expand_units is None else expand_units,
    )
    print(f'Retrieval completed in {time.perf_counter() - search_t0:.2f}s')
    rows = []
    for rank, chunk in enumerate(result.chunks, start=1):
        rows.append({
            'rank': rank,
            'chunk_id': chunk.chunk_id,
            'citation': chunk.citation_anchor or chunk.citation_label,
            'title': chunk.title,
            'unit_type': chunk.unit_type,
            'validity_group': chunk.validity_group,
            'parent_unit_id': chunk.parent_unit_id,
            'vector_score': round(chunk.vector_score, 4),
            'rerank_score': round(chunk.rerank_score, 4),
            'text': chunk.chunk_text[:700],
        })
    return rows, result


def show_results(rows):
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(json.dumps(row, ensure_ascii=False, indent=2))


## 5.1 Benchmark helper

In [ ]:
def benchmark_search(query: str, repeats: int = 3, filter_profile: str = FILTER_PROFILE):
    timings = []
    for i in range(repeats):
        t0 = time.perf_counter()
        rows, result = search(query, top_n=TOP_N, filter_profile=filter_profile)
        elapsed = time.perf_counter() - t0
        timings.append(elapsed)
        print(f'Run {i + 1}/{repeats}: {elapsed:.2f}s, returned={len(result.chunks)}, candidates={result.total_candidates}')
    avg = sum(timings) / len(timings)
    print(f'Average retrieval time over {repeats} runs: {avg:.2f}s')
    return timings


## 6. Run a query

In [ ]:
query_t0 = time.perf_counter()
query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'

# Primary run under default FILTER_PROFILE
rows, result = search(query, top_n=10, filter_profile=FILTER_PROFILE)
print('Filter profile used:', result.filter_profile_used)
print('Total candidates:', result.total_candidates)
print('Empty filter warning:', result.empty_filter_warning)
show_results(rows)
print(f'Query phase completed in {time.perf_counter() - query_t0:.2f}s')

# --- US3: filter-profile comparison (FR-005 / SC-004) ---
print('\n=== Filter profile comparison (same query) ===')
for profile in ('current_law', 'broad', 'historical'):
    _, r = search(query, top_n=TOP_N, filter_profile=profile)
    print(
        f'  {profile:12s} | candidates={r.total_candidates:4d} | '
        f'returned={len(r.chunks):2d} | empty_filter_warning={r.empty_filter_warning}'
    )

# graph_guided is intentionally not run (FR-010)
print('  graph_guided  | not exercised — requires knowledge graph module')

# --- US3: same-provision expansion demo (FR-009) ---
print('\n=== Expansion demo (expand_units=True vs False) ===')
_, r_no = search(query, top_n=5, filter_profile='broad', expand_units=False)
_, r_yes = search(query, top_n=5, filter_profile='broad', expand_units=True)
print(f'  expand_units=False → {len(r_no.chunks)} chunks')
print(f'  expand_units=True  → {len(r_yes.chunks)} chunks')
parent_ids = {c.parent_unit_id for c in r_yes.chunks if c.parent_unit_id}
print(f'  unique parent_unit_id among expanded results: {len(parent_ids)}')


## 7. Optional: inspect one full chunk

In [ ]:
if result.chunks:
    chunk = result.chunks[0]
    print('chunk_id:', chunk.chunk_id)
    print('citation:', chunk.citation_anchor or chunk.citation_label)
    print('title:', chunk.title)
    print('scores:', {'vector': chunk.vector_score, 'rerank': chunk.rerank_score})
    print('--- text ---')
    print(chunk.chunk_text)
    print('--- metadata keys ---')
    print(sorted(chunk.metadata.keys()))


## 8. Configure the answer generator (OpenAI-compatible API)

Set `LLM_BASE_URL`, `LLM_API_KEY`, and `LLM_MODEL_NAME` via environment variables so credentials never end up hardcoded in this notebook. Any OpenAI-compatible chat completions endpoint works (OpenAI, Azure OpenAI, vLLM, Together, OpenRouter, etc.).

```bash
export LLM_BASE_URL="https://api.your-provider.com/v1"
export LLM_API_KEY="..."
export LLM_MODEL_NAME="gpt-4o-mini"
```

If these are not set, the notebook still runs in retrieval-only mode; generation cells will skip cleanly.

**Security (FR-018):** only a masked key is printed. Raw keys are never logged.


In [ ]:
from generation.reasoning_client import GeneratorConfig

generator_config = GeneratorConfig(
    base_url=os.environ.get('LLM_BASE_URL', '').strip(),
    api_key=os.environ.get('LLM_API_KEY', '').strip(),
    model_name=os.environ.get('LLM_MODEL_NAME', '').strip(),
)

# Back-compat aliases used by older cells / mental model
BASE_URL = generator_config.base_url
API_KEY = generator_config.api_key
MODEL_NAME = generator_config.model_name

if not generator_config.is_complete():
    print('Generator not fully configured. Set LLM_BASE_URL, LLM_API_KEY, LLM_MODEL_NAME env vars to enable answer generation.')
else:
    print('Generator configured:')
    print('  BASE_URL:', BASE_URL)
    print('  MODEL_NAME:', MODEL_NAME)
    print('  API_KEY:', generator_config.masked_key())


## 9. Generator client and answer-generation helper

Uses the extracted module `generation.reasoning_client`:
- `GeneratorClient.generate` → `RawGenerationResponse`
- `parse_generation_response` — three shapes: dedicated reasoning field, `<think>...</think>` block, or `not_returned`
- `generate_answer` → `GenerationOutcome` (skip empty context / error / parsed)


In [ ]:
from generation.reasoning_client import (
    ANSWER_PROMPT,
    GenerationOutcome,
    GeneratorClient,
    ParsedAnswer,
    RawGenerationResponse,
    format_context_for_prompt,
    generate_answer as _generate_answer_outcome,
    parse_generation_response,
)

generator: GeneratorClient | None = None
if generator_config.is_complete():
    generator = GeneratorClient(
        base_url=generator_config.base_url,
        api_key=generator_config.api_key,
        model=generator_config.model_name,
    )
    print('Generator client ready.')
    print('ANSWER_PROMPT includes reasoning instruction:', 'reasoning' in ANSWER_PROMPT.lower() or 'suy luận' in ANSWER_PROMPT.lower())
else:
    print('Generator client not created (missing config). Retrieval-only mode.')


def generate_answer(query: str, chunks, *, qa_id: str | None = None) -> GenerationOutcome:
    """Notebook wrapper: returns GenerationOutcome (never raises for empty context)."""
    gen_t0 = time.perf_counter()
    outcome = _generate_answer_outcome(generator, query, chunks, qa_id=qa_id)
    print(f'Generation completed in {time.perf_counter() - gen_t0:.2f}s')
    return outcome


def display_generation_outcome(outcome: GenerationOutcome) -> None:
    """Print answer and reasoning as two distinct sections (FR-020/FR-021)."""
    if outcome.skipped_empty_context:
        print('Skipped generation: empty retrieved context.')
        return
    if outcome.error:
        print('--- Generation error ---')
        print(outcome.error)
        return
    parsed = outcome.parsed
    assert parsed is not None
    print('\n--- Answer ---')
    print(parsed.answer or '(empty)')
    print('\n--- Reasoning ---')
    if parsed.reasoning_available and parsed.reasoning:
        print(f'(source={parsed.reasoning_source})')
        print(parsed.reasoning)
    else:
        print('not returned by this model')


## 10. Full RAG pipeline: retrieve + generate

Ad hoc `ask()` runs retrieval then generation. Final answer and model reasoning are shown as **two distinct sections**. If the model does not return reasoning, the notebook prints `not returned by this model` rather than inventing text.


In [ ]:
def ask(query: str, top_n: int = TOP_N, filter_profile: str = FILTER_PROFILE, score_threshold: float | None = SCORE_THRESHOLD):
    """Run the whole retrieval system end to end: retrieve citation-ready chunks, then generate a grounded answer."""
    rows, result = search(query, top_n=top_n, filter_profile=filter_profile, score_threshold=score_threshold)
    print('Filter profile used:', result.filter_profile_used)
    print('Total candidates:', result.total_candidates)
    print('Empty filter warning:', result.empty_filter_warning)
    show_results(rows)

    if not result.chunks:
        print('No chunks retrieved above the score threshold; skipping generation.')
        outcome = GenerationOutcome(qa_id=None, parsed=None, skipped_empty_context=True, error=None)
        return {'query': query, 'outcome': outcome, 'result': result}

    if generator is None:
        print('Generator not configured; returning retrieval-only result.')
        return {'query': query, 'outcome': None, 'result': result}

    outcome = generate_answer(query, result.chunks)
    display_generation_outcome(outcome)

    print('\n--- Citations used ---')
    for rank, chunk in enumerate(result.chunks, start=1):
        print(f'[{rank}] {chunk.citation_anchor or chunk.citation_label} - {chunk.title}')
    return {'query': query, 'outcome': outcome, 'result': result}


pipeline_query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'
# Uncomment when generator credentials are set (or run retrieval-only with generator=None):
# pipeline_output = ask(pipeline_query, top_n=10, filter_profile='broad')
print('ask() defined. Call: pipeline_output = ask(pipeline_query, top_n=10, filter_profile="broad")')


## 11. Optional: run the full pipeline over a benchmark sample

`run_benchmark_sample` scores retrieval hit-rate against `data/benchmark/qa_final.jsonl` and, when a generator is configured, records per-question `GenerationOutcome` (answer, reasoning source, errors). A single generation failure does **not** stop the loop (FR-023 / SC-008).

This is a demo/validation path — not a full LLM-as-judge evaluation. Use `scripts/evaluate_e2e.py` for judged scoring (FR-024).


In [ ]:
import random


def run_benchmark_sample(
    qa_path: Path | None = None,
    sample_size: int = BENCHMARK_SAMPLE_SIZE,
    filter_profile: str = FILTER_PROFILE,
    seed: int = 42,
    run_generation: bool = True,
):
    """Run retrieval (+ generation, if configured) over a random sample of qa_final.jsonl.

    Reports per-question retrieval hit (whether any ground-truth chunk/provision/document id
    appears among the retrieved results) plus an aggregate hit rate and average latency.
    Unanswerable questions (empty ground_truth) are excluded from the hit-rate denominator.

    Generation reuses already-retrieved chunks (no re-query). Failures are recorded per
    question and do not abort the loop.
    """
    qa_path = qa_path or (PROJECT_ROOT / 'data' / 'benchmark' / 'qa_final.jsonl')
    if not qa_path.exists():
        raise FileNotFoundError(f'Benchmark file not found at {qa_path}')

    with qa_path.open('r', encoding='utf-8') as f:
        all_cases = [json.loads(line) for line in f if line.strip()]

    rng = random.Random(seed)
    sample = rng.sample(all_cases, min(sample_size, len(all_cases)))

    records = []
    latencies = []
    hits = 0
    scored = 0
    gen_errors = 0
    gen_skipped = 0
    gen_ok = 0

    for qa in sample:
        question = qa.get('question') or ''
        ground_truth = qa.get('ground_truth') or {}
        gt_ids = (
            set(ground_truth.get('chunk_ids') or [])
            | set(ground_truth.get('provision_ids') or [])
            | set(ground_truth.get('document_ids') or [])
        )
        is_unanswerable = qa.get('answer_type') == 'unanswerable' or not gt_ids

        t0 = time.perf_counter()
        _, result = search(question, top_n=TOP_N, filter_profile=filter_profile)
        elapsed = time.perf_counter() - t0
        latencies.append(elapsed)

        retrieved_ids = set()
        for chunk in result.chunks:
            retrieved_ids.update({chunk.chunk_id, chunk.parent_unit_id, chunk.id_str})

        hit = bool(gt_ids & retrieved_ids)
        if not is_unanswerable:
            scored += 1
            if hit:
                hits += 1

        outcome: GenerationOutcome | None = None
        if run_generation and generator is not None:
            # generate_answer never raises for empty context; API errors become outcome.error
            outcome = _generate_answer_outcome(
                generator,
                question,
                result.chunks,
                qa_id=qa.get('qa_id'),
            )
            if outcome.skipped_empty_context:
                gen_skipped += 1
            elif outcome.error:
                gen_errors += 1
            elif outcome.parsed is not None:
                gen_ok += 1

        parsed = outcome.parsed if outcome else None
        records.append({
            'qa_id': qa.get('qa_id'),
            'question': question,
            'answer_type': qa.get('answer_type'),
            'category': qa.get('category'),
            'is_unanswerable': is_unanswerable,
            'retrieval_hit': hit,
            'total_candidates': result.total_candidates,
            'latency_s': round(elapsed, 3),
            'generated_answer': parsed.answer if parsed else None,
            'reasoning': parsed.reasoning if parsed else None,
            'reasoning_source': parsed.reasoning_source if parsed else None,
            'reasoning_available': parsed.reasoning_available if parsed else False,
            'generation_error': outcome.error if outcome else None,
            'skipped_empty_context': outcome.skipped_empty_context if outcome else False,
        })

    hit_rate = hits / scored if scored else float('nan')
    avg_latency = sum(latencies) / len(latencies) if latencies else float('nan')

    print(f'Sampled {len(sample)} questions ({scored} scored, {len(sample) - scored} unanswerable excluded)')
    print(f'Hit rate: {hit_rate:.2%}' if scored else 'Hit rate: n/a (no scored questions)')
    print(f'Average retrieval latency: {avg_latency:.3f}s')
    if run_generation and generator is not None:
        print(f'Generation: ok={gen_ok}, skipped_empty={gen_skipped}, errors={gen_errors}')

    return {
        'records': records,
        'hit_rate': hit_rate,
        'avg_latency_s': avg_latency,
        'sample_size': len(sample),
        'scored': scored,
        'generation_ok': gen_ok,
        'generation_errors': gen_errors,
        'generation_skipped': gen_skipped,
    }


# Uncomment to run (generation requires LLM_* env vars; retrieval-only works without them):
# benchmark_summary = run_benchmark_sample(sample_size=BENCHMARK_SAMPLE_SIZE, run_generation=True)
print('run_benchmark_sample() defined. Example: benchmark_summary = run_benchmark_sample(sample_size=10)')
